In [1]:
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
from PIL import Image
from sklearn.metrics import f1_score
from transformers import CLIPProcessor, CLIPModel, AutoModel, AutoImageProcessor

resnet = models.resnet50()

vgg19 = models.vgg19()

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

dinov2_vitb14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')

dinov3_model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"
dinov3_processor = AutoImageProcessor.from_pretrained(dinov3_model_name)
dinov3_model = AutoModel.from_pretrained(
    dinov3_model_name, 
    device_map="auto", 
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Using cache found in /Users/osannadeng/.cache/torch/hub/facebookresearch_dinov2_main
/Users/osannadeng/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Us

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

## CLIP

In [2]:
# split 
train_images = []
train_labels = []
train_data_path = os.path.join("..", "data", "CitrusUAT_split_images", "train")
for root, dirs, files in os.walk(train_data_path):
    if len(dirs) != 0:
        labels = dirs
    else:
        # imgs = [{"image": file, "label": label} for file in files]
        label = root.replace(train_data_path, "").replace("_", " ")
        label = label.replace("Fe", "Iron Deficiency").replace("HLB", "Huanglongbing").replace("Mg", "Magnesium Deficiency").replace("Mn", "Manganese Deficiency").replace("N", "Nitrogen Deficiency").replace("Zn", "Zinc Deficiency")
        train_images.extend([os.path.join(root, file) for file in files])
        train_labels.extend([label] * len(files))

val_images = []
val_labels = []
val_data_path = os.path.join("..", "data", "CitrusUAT_split_images", "val")
for root, dirs, files in os.walk(val_data_path):
    if len(dirs) != 0:
        labels = dirs
    else:
        label = root.replace(val_data_path, "").replace("_", " ")
        label = label.replace("Fe", "Iron Deficiency").replace("HLB", "Huanglongbing").replace("Mg", "Magnesium Deficiency").replace("Mn", "Manganese Deficiency").replace("N", "Nitrogen Deficiency").replace("Zn", "Zinc Deficiency")
        val_images.extend([os.path.join(root, file) for file in files])
        val_labels.extend([label] * len(files))

test_images = []
test_labels = []
test_data_path = os.path.join("..", "data", "CitrusUAT_split_images", "test")
for root, dirs, files in os.walk(test_data_path):
    if len(dirs) != 0:
        labels = dirs
    else:
        label = root.replace(test_data_path, "").replace("_", " ")
        label = label.replace("Fe", "Iron Deficiency").replace("HLB", "Huanglongbing").replace("Mg", "Magnesium Deficiency").replace("Mn", "Manganese Deficiency").replace("N", "Nitrogen Deficiency").replace("Zn", "Zinc Deficiency")
        test_images.extend([os.path.join(root, file) for file in files])
        test_labels.extend([label] * len(files))

# labels = [label.replace(train_data_path, "").replace("_", " ").replace("Fe", "Iron Deficiency").replace("HLB", "Huanglongbing").replace("Mg", "Magnesium Deficiency").replace("Mn", "Manganese Deficiency").replace("N", "Nitrogen Deficiency").replace("Zn", "Zinc Deficiency") for label in labels]

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
clip_model = clip_model.to(device)
clip_model.float()

class_prompts = sorted(list(set(train_labels)))
class_to_idx = {t: i for i, t in enumerate(class_prompts)}

with torch.no_grad():
    text_inputs = clip_processor(
        text=class_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

    # text_outputs = clip_model.text_model(
    #     input_ids=text_inputs["input_ids"],
    #     attention_mask=text_inputs["attention_mask"],
    #     return_dict=True
    # )
    text_feats = clip_model.get_text_features(**text_inputs)
    if not torch.is_tensor(text_feats):
        text_feats = text_feats.pooler_output

    class_text_embeds = F.normalize(text_feats, dim=-1)

# create dataset
class clipdataset():
    def __init__(self, image_paths, labels):
        self.image_paths = image_paths
        self.labels = labels
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        return image, label

def collate_fn(batch):
    images, texts = zip(*batch)
    inputs = clip_processor(
        text=list(texts),
        images=list(images),
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    inputs["raw_texts"] = list(texts)
    return inputs

clip_train_dataloader = DataLoader(clipdataset(train_images, train_labels), batch_size=32, shuffle=True, collate_fn=collate_fn)

clip_val_dataloader = DataLoader(clipdataset(val_images, val_labels), batch_size=64, shuffle=False, collate_fn=collate_fn)

clip_test_dataloader = DataLoader(clipdataset(test_images, test_labels), batch_size=64, shuffle=False, collate_fn=collate_fn)

# loss function + num of correct guesses
def clip_criterion(image_embeds, text_embeds, logit_scale):
    # normalize
    image_embeds = F.normalize(image_embeds, p=2, dim=-1)
    text_embeds = F.normalize(text_embeds, p=2, dim=-1)

    logits_per_image = logit_scale * (image_embeds @ text_embeds.T)
    logits_per_text = logits_per_image.T

    batch_size = image_embeds.size(0)
    labels = torch.arange(batch_size, device=image_embeds.device)

    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)

    return (loss_i + loss_t) / 2

lr = 1e-5
optimizer = optim.Adam(clip_model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-6, weight_decay=0.01)

num_epochs = 50
import wandb

run = wandb.init(
    entity="rchan192-university-of-california-riverside",
    project="my-awesome-project",
    config={
        "learning_rate": lr,
        "architecture": "CLIP",
        "dataset": "CitrusUAT",
        "epochs": num_epochs,
    },
)

# train
def clip_train(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        
        all_preds = []
        all_targets = []

        for batch in train_loader:
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_loss=False
            )

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            logit_scale = model.logit_scale.exp().clamp(1, 100)
            loss = criterion(image_embeds, text_embeds, logit_scale)

            class_logits = F.normalize(image_embeds, dim=-1) @ class_text_embeds.T
            preds = class_logits.argmax(dim=-1)
            targets = torch.tensor([class_to_idx[t] for t in batch["raw_texts"]], device=device)
            correct = (preds == targets).float()

            total_loss += loss.item() * len(pixel_values)
            total_correct += correct.sum().item()

            all_preds.append(preds.detach().cpu())
            all_targets.append(targets.detach().cpu())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        avg_loss = total_loss / len(train_loader.dataset)
        avg_acc = total_correct / len(train_loader.dataset)

        all_preds = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()
        f1 = f1_score(all_targets, all_preds, average='macro')

        # validation set
        model.eval()
        total_loss_v = 0
        total_correct_v = 0

        all_preds_v = []
        all_targets_v = []
        
        with torch.no_grad():
            for batch in val_loader:
                pixel_values = batch['pixel_values'].to(device)
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
            
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    return_loss=False
                )

                image_embeds = outputs.image_embeds
                text_embeds = outputs.text_embeds

                logit_scale = model.logit_scale.exp().clamp(1, 100)
                loss = criterion(image_embeds, text_embeds, logit_scale)

                class_logits = F.normalize(image_embeds, dim=-1) @ class_text_embeds.T
                preds = class_logits.argmax(dim=-1)
                targets = torch.tensor([class_to_idx[t] for t in batch["raw_texts"]], device=device)
                correct = (preds == targets).float()

                total_loss_v += loss.item() * len(pixel_values)
                total_correct_v += correct.sum().item()
                
                all_preds_v.append(preds.detach().cpu())
                all_targets_v.append(targets.detach().cpu())
        avg_loss_v = total_loss_v / len(val_loader.dataset)
        avg_acc_v = total_correct_v / len(val_loader.dataset)

        all_preds_v = torch.cat(all_preds_v).numpy()
        all_targets_v = torch.cat(all_targets_v).numpy()
        f1_v = f1_score(all_targets_v, all_preds_v, average='macro')
        
        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {avg_loss:.4f}, Training Accuracy: {avg_acc:.4f}, Training F1 Score: {f1:.4f}, Validation Loss: {avg_loss_v:.4f}, Validation Accuracy: {avg_acc_v:.4f}, Validation F1 Score: {f1_v:.4f}")
        run.log({"train loss": avg_loss,"train acc": avg_acc, "train f1 score": f1, "val loss": avg_loss_v, "val acc": avg_acc_v, "val f1 score": f1_v})

def clip_evaluate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0

    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
        
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_loss=False
            )

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            logit_scale = model.logit_scale.exp().clamp(1, 100)
            loss = criterion(image_embeds, text_embeds, logit_scale)

            class_logits = F.normalize(image_embeds, dim=-1) @ class_text_embeds.T
            preds = class_logits.argmax(dim=-1)
            targets = torch.tensor([class_to_idx[t] for t in batch["raw_texts"]], device=device)
            correct = (preds == targets).float()

            total_loss += loss.item() * len(pixel_values)
            total_correct += correct.sum().item()

            all_preds.append(preds.detach().cpu())
            all_targets.append(targets.detach().cpu())
    avg_loss = total_loss / len(test_loader.dataset)
    avg_acc = total_correct / len(test_loader.dataset)

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    f1 = f1_score(all_targets, all_preds, average='macro')
    
    print(f"Final Average Loss: {avg_loss:.4f}, Final Average Accuracy: {avg_acc:.4f}, Final F1 Score: {f1:.4f}")

clip_train(clip_model, clip_train_dataloader, clip_val_dataloader, clip_criterion, optimizer, num_epochs, device)
clip_evaluate(clip_model, clip_test_dataloader, clip_criterion, device)

run.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/osannadeng/.netrc.
wandb: Currently logged in as: odeng002 (rchan192-university-of-california-riverside) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/50, Training Loss: 2.5001, Training Accuracy: 0.2743, Training F1 Score: 0.2380, Validation Loss: 2.9620, Validation Accuracy: 0.4248, Validation F1 Score: 0.3770
Epoch 2/50, Training Loss: 1.4924, Training Accuracy: 0.5302, Training F1 Score: 0.4619, Validation Loss: 2.6906, Validation Accuracy: 0.5556, Validation F1 Score: 0.4946
Epoch 3/50, Training Loss: 1.3346, Training Accuracy: 0.6549, Training F1 Score: 0.5643, Validation Loss: 2.7131, Validation Accuracy: 0.6144, Validation F1 Score: 0.5526
Epoch 4/50, Training Loss: 1.2834, Training Accuracy: 0.6667, Training F1 Score: 0.5895, Validation Loss: 2.6342, Validation Accuracy: 0.6013, Validation F1 Score: 0.5682
Epoch 5/50, Training Loss: 1.2564, Training Accuracy: 0.7034, Training F1 Score: 0.6232, Validation Loss: 2.6307, Validation Accuracy: 0.5621, Validation F1 Score: 0.5124
Epoch 6/50, Training Loss: 1.3016, Training Accuracy: 0.6890, Training F1 Score: 0.6104, Validation Loss: 2.6257, Validation Accuracy: 0.5948, Va

train acc,▁▄▅▅▅▆▆▆▆▆▇▇▆▇▇▇▇▇▇████▇▇▇▇██▇▇▇▇▇██████
train f1 score,▁▄▅▅▅▆▆▆▆▅▅▇▇▆▇▇▇▇▇▇▇████▇▇▇██▇▇▇▇█████▇
train loss,█▂▂▁▁▂▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁
val acc,▁▃▃▃▃▄▅▅▄▃▆▆▆▆▇▆▆▆▇▇▇▇▇▆█▆███▇▇▇▇▇▇███▆▆
val f1 score,▁▃▃▄▃▄▆▅▄▃▆▆▆▇▇▆▆▇▇▇▇▇▇█▆███▇▇▇▇▇▇████▅▆
val loss,█▃▃▂▂▂▁▁▁▂▂▂▂▁▁▁▁▁▁▁▁▁▁▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▃▂
train acc,0.89633
train f1 score,0.81499
train loss,1.31033
val acc,0.81699
val f1 score,0.72638
